# Li Auto Sales Forecaster

End-to-end pipeline: fetch → features → model → evaluate → visualize.

**Feature window:** months 1–3 | **Target window:** months 4–12 | **Split:** cohort (L8/L9/MEGA train, L6/L7 test)

In [ ]:
import sys, os
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
import numpy as np

## 1. Load Features and Targets

In [ ]:
features = pd.read_parquet('../data/processed/features.parquet')
target = pd.read_parquet('../data/processed/target.parquet')
print(features.shape, target.shape)
features.head()

## 2. Train / Test Split (cohort, not random)

In [ ]:
from src.model.split import cohort_split
X_train, X_test, y_train, y_test = cohort_split(features, target)
print('Train:', X_train['model'].tolist())
print('Test: ', X_test['model'].tolist())

## 3. Train Quantile Forest

In [ ]:
from src.model.train import QuantileForestForecaster, FEATURE_COLS
qf = QuantileForestForecaster(quantiles=[0.1, 0.5, 0.9], n_estimators=200)
qf.fit(X_train[FEATURE_COLS], y_train)
print('Model trained.')

## 4. Evaluate

In [ ]:
from src.model.evaluate import pinball_loss, mape, interval_coverage
preds = qf.predict(X_test[FEATURE_COLS])
y_true = y_test.values
print(f'Pinball q=0.5: {pinball_loss(y_true.ravel(), preds[:,:,1].ravel(), 0.5):.1f}')
print(f'MAPE:          {mape(y_true.ravel(), preds[:,:,1].ravel()):.1f}%')
print(f'80% coverage:  {interval_coverage(y_true.ravel(), preds[:,:,0].ravel(), preds[:,:,2].ravel()):.2%}')

## 5. Fan Chart — Forecast vs Actuals

In [ ]:
from src.viz.plots import fan_chart
for i, model in enumerate(X_test['model'].tolist()):
    fig = fan_chart(
        preds[i],
        pd.DataFrame({'month_since_launch': range(4,13), 'sales': y_test.values[i]}),
        model_name=model,
        quantiles=[0.1, 0.5, 0.9]
    )
    fig.show()

## 6. Incremental-Value Test

Does each feature group add signal beyond the other two?

In [ ]:
from src.model.evaluate import incremental_value_test
traj_cols = [c for c in FEATURE_COLS if 'percentile' in c or 'ratio' in c]
spec_cols = [c for c in FEATURE_COLS if c.startswith('pc') or c == 'is_erev']
sent_cols = [c for c in FEATURE_COLS if c.startswith('sent_')]
groups = {
    'trajectory': X_train[traj_cols].values,
    'specs': X_train[spec_cols].values,
    'sentiment': X_train[sent_cols].values,
}
results = incremental_value_test(groups, y_train.values)
for g, m in results.items():
    direction = 'adds signal' if m['delta_pinball'] < 0 else 'no added signal'
    print(f'{g:12s}  delta_pinball={m["delta_pinball"]:+.1f}  ({direction})')

## 7. Geographic Choropleth

Requires `data/processed/regional.parquet` with columns [model, province, month_since_launch, sales].

In [ ]:
import os
if os.path.exists('../data/processed/regional.parquet'):
    from src.viz.choropleth import plot_choropleth, regional_ranking_table
    regional = pd.read_parquet('../data/processed/regional.parquet')
    plot_choropleth(regional, model='L9').show()
    print(regional_ranking_table(regional, model='L9').to_string())
else:
    print('No regional data available. Populate data/processed/regional.parquet from CPCA province breakdowns.')